# Fine-tuning Laya on Chess "Typed Decisions"

This notebook fine-tunes **Laya** (`convaiinnovations/laya`, a ModernBERT-large based decision agent) to choose chess moves in two stages: first the **piece** to move, then its **destination square**. Each stage is a multiple-choice question about a textual board state, and the training target is a *probability distribution* over the options rather than a single hard label.

## Pipeline

| Step | What happens | Produces |
|---|---|---|
| 1 | Verify the 2×T4 GPU environment | – |
| 2 | Install `laya`, load tokenizer and config | `tok`, `cfg`, `model_dir` |
| 3 | Load the JSONL dataset, sanity-check schema and duplicates | `rows` |
| 4 | Convert rows into tokenised Laya training items | `examples` |
| 5 | Split into train / validation / test | `split_indices.json` |
| 6 | Evaluate the **base** model on held-out test rows | baseline results JSON |
| 7 | Save train / validation items to disk | `train_items.pt`, `validation_items.pt` |
| 8 | Write the DDP fine-tuning script | `train_ddp.py` |
| 9 | Launch fine-tuning on both GPUs | `laya_finetuned_typed_decisions/` |

**Requirements:** Kaggle notebook with **GPU T4 ×2**, internet on (for `pip` and the Hugging Face download), and the dataset notebook output `chess-dataset-etl-for-laya-typed-decision` attached as input. Run the cells top to bottom.

## Configuration

Everything you might want to change lives in this one cell: model id, dataset path, how many rows to use, split ratios and every output path. The rest of the notebook only reads these values.

In [1]:
from pathlib import Path

# --- Model -------------------------------------------------------------
MODEL_ID = "convaiinnovations/laya"

# --- Data --------------------------------------------------------------
JSONL_PATH = (
    "/kaggle/input/notebooks/itsanshuguleria/"
    "chess-dataset-etl-for-laya-typed-decision/laya_chess_training.jsonl"
)
MAX_ROWS = None  # None = use every row; set e.g. 1000 for a quick smoke test

# --- Train / validation / test split -----------------------------------
SPLIT_SEED = 42
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1  # the remainder becomes the test set

# --- Baseline evaluation -----------------------------------------------
BASELINE_N = 1000  # number of test rows used to score the base model

# --- Output paths ------------------------------------------------------
WORK_DIR = Path("/kaggle/working")
SPLIT_PATH = WORK_DIR / "split_indices.json"
BASELINE_RESULTS_PATH = WORK_DIR / f"laya_baseline_{BASELINE_N}_before_finetuning.json"
TRAIN_ITEMS_PATH = WORK_DIR / "train_items.pt"
VAL_ITEMS_PATH = WORK_DIR / "validation_items.pt"
TRAIN_SCRIPT_PATH = WORK_DIR / "train_ddp.py"
OUTPUT_DIR = WORK_DIR / "laya_finetuned_typed_decisions"

## Step 1 — Verify the environment

Fine-tuning uses PyTorch **DistributedDataParallel (DDP)** across two GPUs, so this cell fails fast if CUDA is missing or fewer than two GPUs are visible. In Kaggle choose *Settings → Accelerator → GPU T4 ×2*.

It also sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, which reduces memory fragmentation on the 15 GB T4 cards. The training processes started later by `torchrun` inherit this variable.

In [2]:
# MAGIC# !nvidia-smi

import os
import sys

import torch

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__} (CUDA {torch.version.cuda})")

n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}   : {props.name} | {props.total_memory / 1024**3:.1f} GB")

assert torch.cuda.is_available(), "CUDA is not available."
assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}. "
    "In Kaggle: Notebook options → Accelerator → GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("\n✓ Two GPUs detected, environment ready.")

Python  : 3.13.15
PyTorch : 2.11.0+cu128 (CUDA 12.8)
GPU 0   : Tesla T4 | 14.6 GB
GPU 1   : Tesla T4 | 14.6 GB

✓ Two GPUs detected, environment ready.


## Step 2 — Install Laya and load the base checkpoint

`laya` provides the `Agent` class used for inference plus the helpers needed to build training data (`build_sequence`, `render_options`) and the model (`build_model`, `proper_reward`).

The next two cells:

1. install the package,
2. download the base checkpoint snapshot from the Hugging Face Hub and load what the data pipeline needs: the **tokenizer** and the **config** (`rl_agent_config.json`).

`cfg["max_len"]` and `cfg["head_max_len"]` control how long the tokenised sequences may be, and `QTYPES` maps question types (`choice`, `score`, `noul`) to the integer ids stored in each training item. `_fix_tokenizer_config` applies the tokenizer-config fix that Laya's own reference notebook uses.

In [3]:
!pip install -q -U laya

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.7/312.7 kB 18.1 MB/s eta 0:00:00


In [4]:
import json

import laya
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from laya.agent import _fix_tokenizer_config
from laya.common import QTYPES, build_sequence, render_options

print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

print("\nModel dir :", model_dir)
print(
    f"Tokenizer : {type(tok).__name__} | vocab {len(tok)} | "
    f"pad {tok.pad_token!r} (id {tok.pad_token_id})"
)
print("Question types:", QTYPES)
print("\nLaya config:")
for key, value in cfg.items():
    print(f"  {key}: {value}")

Fetching tokenizer and config from convaiinnovations/laya...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 39 files:   0%|          | 0/39 [00:00<?, ?it/s]


Model dir : /root/.cache/huggingface/hub/models--convaiinnovations--laya/snapshots/7b928d828b7b0e022f929d9bd2e44165aa270148
Tokenizer : TokenizersBackend | vocab 50368 | pad '[PAD]' (id 50283)
Question types: {'choice': 0, 'score': 1, 'noul': 2}

Laya config:
  encoder: answerdotai/ModernBERT-large
  head_layers: 2
  max_len: 512
  head_max_len: 192
  max_prefixes: 6
  act_costs: {'escalate': 0.5}
  cost_wrong_act: 3.0
  amp_dtype: bf16
  model_name: rl-agent
  temperature: [1.6369030475616455, 1.2514300346374512, 1.983399510383606]
  temperature_by_options: {'choice:3-5': 1.7601518630981445, 'choice:6-10': 1.0000158548355103, 'score:3-5': 1.2514300346374512, 'noul:2': 1.983399510383606, 'choice:11+': 0.10058280825614929, 'choice:2': 1.9063563346862793}
  training: {'updates': 7313, 'epochs_completed': 1, 'hours': 1.96, 'world_size': 1, 'fine_tuned_from_checkpoint': True}


## Step 3 — Load and validate the dataset

The dataset is a JSONL file with one **typed decision** per line. Each row has three fields:

- `state` – text describing the position and the current stage (`piece` or `destination`),
- `questions` – exactly one `choice` question with its `instructions` and the available options (`criteria`),
- `gold` – the target probability for every option.

```json
{
  "state": "Chess position. FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1. Side to move: white. Stage: piece.",
  "questions": {"piece_selection": {"type": "choice",
                 "instructions": "Choose the best chess piece for the move.",
                 "criteria": {"P": "Pawn", "N": "Knight"}}},
  "gold": {"probabilities": {"P": 0.842, "N": 0.158}}
}
```

The file is read **once** into `rows`; every later step reuses that list instead of re-reading the file.

In [5]:
import hashlib
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


def load_jsonl(path):
    """Read a JSONL file into a list of dicts, skipping blank lines."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}") from e
    return rows


rows = load_jsonl(JSONL_PATH)
df = pd.DataFrame(rows)  # top level only; questions / gold stay as dicts

print(f"Rows loaded  : {len(rows):,}")
print(f"DataFrame    : {df.shape} | columns: {df.columns.tolist()}")

Rows loaded  : 150,382
DataFrame    : (150382, 3) | columns: ['state', 'questions', 'gold']


### Inspect a few rows

In [6]:
N_SAMPLES = 3

for i, row in enumerate(rows[:N_SAMPLES]):
    print(f"\n{'=' * 80}\nSAMPLE {i}\n{'=' * 80}")
    print(json.dumps(row, indent=2, ensure_ascii=False))


SAMPLE 0
{
  "state": "Chess position. FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1. Side to move: white. Stage: piece.",
  "questions": {
    "piece_selection": {
      "type": "choice",
      "instructions": "Choose the best chess piece for the move.",
      "criteria": {
        "P": "Pawn",
        "N": "Knight"
      }
    }
  },
  "gold": {
    "probabilities": {
      "P": 0.84233856376144,
      "N": 0.15766143623856002
    }
  }
}

SAMPLE 1
{
  "state": "Chess position. FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1. Side to move: white. Stage: destination. Selected piece: P.",
  "questions": {
    "destination_selection": {
      "type": "choice",
      "instructions": "Choose the best chess piece destination.",
      "criteria": {
        "h3": "h3",
        "g3": "g3",
        "f3": "f3",
        "e3": "e3",
        "d3": "d3",
        "c3": "c3",
        "b3": "b3",
        "a3": "a3",
        "h4": "h4",
        "g4": "g4",
        "f4": "

### Check the schema of every row

Training silently degrades if a row has missing options or a broken distribution, so each row is validated against these rules:

- exactly the keys `state`, `questions`, `gold`;
- exactly one question, of type `choice`, with at least one option;
- every option has a gold probability that is non-negative, and the probabilities sum to more than 0.

Failures are grouped by error message so a systematic problem is easy to spot.

In [7]:
EXPECTED_KEYS = {"state", "questions", "gold"}


def validate_row(row):
    """Raise if `row` does not follow the typed-decision schema."""
    assert set(row) == EXPECTED_KEYS, f"unexpected keys: {sorted(row)}"
    assert isinstance(row["state"], str), "state must be a string"
    assert isinstance(row["questions"], dict), "questions must be a dict"
    assert isinstance(row["gold"], dict), "gold must be a dict"
    assert len(row["questions"]) == 1, "expected exactly one question"

    q = next(iter(row["questions"].values()))
    assert isinstance(q, dict) and q["type"] == "choice", "question must be a choice"
    criteria = q["criteria"]
    assert isinstance(criteria, dict) and criteria, "criteria must be a non-empty dict"

    probs = row["gold"]["probabilities"]
    assert isinstance(probs, dict), "gold probabilities must be a dict"

    missing = [k for k in criteria if k not in probs]
    if missing:
        raise ValueError(f"missing gold probabilities: {missing}")

    values = [float(probs[k]) for k in criteria]
    if min(values) < 0:
        raise ValueError("negative probability")
    if sum(values) <= 0:
        raise ValueError(f"probability sum <= 0: {sum(values)}")


error_counts = Counter()
bad_rows = []  # (row index, error, preview) for the first few failures

for idx, row in enumerate(rows):
    try:
        validate_row(row)
    except Exception as e:
        error = f"{type(e).__name__}: {str(e).strip() or '<no message>'}"
        error_counts[error] += 1
        if len(bad_rows) < 20:
            bad_rows.append((idx, error, json.dumps(row, ensure_ascii=False)[:500]))

print(f"Invalid rows: {sum(error_counts.values()):,}")
for error, count in error_counts.most_common():
    print(f"  {count:6,} : {error}")

for idx, error, preview in bad_rows:
    print(f"\nRow {idx}\n  Error: {error}\n  Data : {preview}")

if not error_counts:
    print("✓ All rows follow the expected schema")

Invalid rows: 0
✓ All rows follow the expected schema


### Check for exact duplicates

Each row is serialised to canonical JSON (sorted keys) and hashed. Identical rows that end up in both the train and test split would inflate the evaluation, so any duplicates found here should be removed before continuing.

In [8]:
groups = defaultdict(list)  # hash of canonical JSON -> row indices

for idx, row in enumerate(rows):
    canonical = json.dumps(
        row, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    )
    groups[hashlib.sha256(canonical.encode("utf-8")).hexdigest()].append(idx)

duplicate_groups = [g for g in groups.values() if len(g) > 1]
extra_rows = sum(len(g) - 1 for g in duplicate_groups)

print(f"Total rows               : {len(rows):,}")
print(f"Unique complete JSONs    : {len(groups):,}")
print(f"Duplicate groups         : {len(duplicate_groups):,}")
print(f"Extra duplicate rows     : {extra_rows:,}")

for n, group in enumerate(duplicate_groups[:20], start=1):
    print(f"\nDuplicate group {n}: rows {group}")
    print(json.dumps(rows[group[0]], indent=2, ensure_ascii=False))

if not duplicate_groups:
    print("\n✓ No complete duplicate JSON objects found")

Total rows               : 150,382
Unique complete JSONs    : 150,382
Duplicate groups         : 0
Extra duplicate rows     : 0

✓ No complete duplicate JSON objects found


## Step 4 — Convert rows into Laya training examples

Laya does not read raw JSON. Each row is turned into a **training item** with these fields:

| Field | Meaning |
|---|---|
| `ids` | token ids of the state + instructions + rendered options (truncated to `cfg["max_len"]`) |
| `markers` | token positions of the option markers, one per answer option |
| `qtype` | integer id of the question type (`QTYPES["choice"]` = 0) |
| `target` | gold probabilities in option order, normalised to sum to 1 |
| `label` | index of the most likely option (hard label) |

Two safety checks are built in:

- if the gold probabilities sum to 0, the target falls back to a uniform distribution;
- if the number of markers returned by `build_sequence` differs from the number of rendered options (for example because truncation cut an option off), the row is rejected and counted as **failed**.

`normalize_probs` is defined once here and reused when scoring models in Step 6.

Set `MAX_ROWS` in the configuration cell to build only the first N rows for a quick test.

In [9]:
def normalize_probs(prob_dict, keys):
    """Probabilities in the order of `keys`, rescaled to sum to 1 (uniform if all 0)."""
    probs = np.array([float(prob_dict.get(k, 0.0)) for k in keys], dtype=np.float64)
    total = probs.sum()
    if total > 0:
        return probs / total
    return np.full(len(keys), 1.0 / len(keys))


def build_training_item(row, tok, cfg):
    """Convert one JSONL row to a Laya training item, or None on marker mismatch."""
    qid, q = next(iter(row["questions"].items()))  # one question per row
    qtype, crit = q["type"], q["criteria"]

    target = normalize_probs(row["gold"]["probabilities"], list(crit)).tolist()
    label = int(np.argmax(target))

    n_options = len(render_options({"t": qtype, "crit": crit}))
    seq, markers = build_sequence(
        tok,
        row["state"],
        {"t": qtype, "ins": q["instructions"], "crit": crit},
        cfg["max_len"],
        cfg["head_max_len"],
    )
    if len(markers) != n_options:
        return None

    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[qtype],
        "target": target,
        "label": label,
    }


def build_examples(rows, tok, cfg):
    """Build items for all rows; return (examples, failed)."""
    examples, failed = [], []
    for row_idx, row in enumerate(rows):
        try:
            item = build_training_item(row, tok, cfg)
        except Exception as e:
            failed.append(
                {"row": row_idx, "error_type": type(e).__name__, "error": repr(e)}
            )
            continue
        if item is None:
            failed.append({"row": row_idx, "reason": "marker_count_mismatch"})
        else:
            examples.append(item)
    return examples, failed

In [10]:
rows_in_use = rows if MAX_ROWS is None else rows[:MAX_ROWS]
examples, failed = build_examples(rows_in_use, tok, cfg)

print(f"Input rows        : {len(rows_in_use):,}")
print(f"Training examples : {len(examples):,}")
print(f"Failed            : {len(failed):,}")

if failed:
    print("\nFirst 20 failures:")
    for x in failed[:20]:
        print(" ", x)
else:
    print("\n✓ All rows converted to Laya examples")

if examples:
    ex = examples[0]
    print("\nFirst example")
    print("  ids length    :", len(ex["ids"]))
    print("  markers       :", ex["markers"])
    print("  qtype         :", ex["qtype"])
    print("  target length :", len(ex["target"]))
    print("  target sum    :", sum(ex["target"]))
    print("  label         :", ex["label"])

Input rows        : 150,382
Training examples : 150,382
Failed            : 0

✓ All rows converted to Laya examples

First example
  ids length    : 81
  markers       : [14, 19]
  qtype         : 0
  target length : 2
  target sum    : 1.0
  label         : 0


## Step 5 — Train / validation / test split

The examples are shuffled with a fixed seed and divided **80 % / 10 % / 10 %**. The test split is never used for training or calibration; it is only used to compare the model before and after fine-tuning.

The indices are written to `split_indices.json` and **re-used if that file already exists**, so re-running the notebook (or restarting the Kaggle session with the file kept) cannot silently change which rows are held out.

> **Notes**
> - The split indices refer to positions in `rows` / `examples`, which line up only when no row failed conversion; the cell asserts this.
> - If you already fine-tuned with an earlier split, put those indices into `split_indices.json` (keys `train`, `val`, `test`) before running this cell. Otherwise the new test rows may overlap with data that model was trained on.
> - The split is at row level. Rows describing the same board position (for example its `piece` and `destination` rows) can fall into different splits, which can make test scores look better than they would on unseen positions. Grouping by FEN would give a stricter split.

In [11]:
assert (
    not failed
), "Some rows failed conversion, so split indices would not match `rows`."
n_total = len(examples)

if SPLIT_PATH.exists():
    split = json.loads(SPLIT_PATH.read_text())
    train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]
    assert len(train_idx) + len(val_idx) + len(test_idx) == n_total, (
        f"{SPLIT_PATH} does not match the current data size ({n_total:,}); "
        "delete it to create a new split."
    )
    print(f"Loaded existing split from {SPLIT_PATH}")
else:
    perm = np.random.default_rng(SPLIT_SEED).permutation(n_total).tolist()
    n_train = int(n_total * TRAIN_FRAC)
    n_val = int(n_total * VAL_FRAC)
    train_idx = perm[:n_train]
    val_idx = perm[n_train : n_train + n_val]
    test_idx = perm[n_train + n_val :]
    SPLIT_PATH.write_text(
        json.dumps(
            {"seed": SPLIT_SEED, "train": train_idx, "val": val_idx, "test": test_idx}
        )
    )
    print(f"Created new split (seed {SPLIT_SEED}) -> {SPLIT_PATH}")

assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)

print(f"Total      : {n_total:,}")
print(f"Train      : {len(train_idx):,}")
print(f"Validation : {len(val_idx):,}")
print(f"Test       : {len(test_idx):,}")
print("✓ No overlap between train / validation / test")

Created new split (seed 42) -> /kaggle/working/split_indices.json
Total      : 150,382
Train      : 120,305
Validation : 15,038
Test       : 15,039
✓ No overlap between train / validation / test


## Step 6 — Baseline: score the base model before fine-tuning

To know whether fine-tuning helped, the **un-tuned** Laya model is first evaluated on the first `BASELINE_N` test rows. For every row the model predicts a probability distribution over the options, which is compared with the gold distribution *p* (prediction) vs *g* (gold):

| Metric | Definition | Better |
|---|---|---|
| Accuracy | predicted choice equals the gold choice (`argmax g`) | higher |
| Soft accuracy | Σ p·g, the probability mass placed where the gold puts it | higher |
| Brier score | Σ (p − g)² | lower |
| TV distance | ½ Σ \|p − g\| | lower |
| KL divergence | Σ g · log(g / p), with clipping for numerical safety | lower |
| Latency | wall-clock time of each `predict` call (mean, p50, p95) | lower |

`evaluate_agent` is written as a reusable function so exactly the same evaluation can be run on the fine-tuned model later. Full per-row predictions are saved next to the summary metrics.

In [12]:
import gc
import time


def evaluate_agent(agent, eval_rows, row_indices, model_name):
    """Run `agent` on each row; return (metrics, per-row predictions)."""
    predictions = []
    soft_acc, brier, tv, kl, latencies = [], [], [], [], []
    t_start = time.time()

    for n, (row_idx, row) in enumerate(zip(row_indices, eval_rows), start=1):
        questions = row["questions"]
        qid, qdef = next(iter(questions.items()))  # one question per row
        assert qdef["type"] == "choice"
        keys = list(qdef["criteria"])

        t0 = time.perf_counter()
        result = agent.predict(row["state"], questions)
        latency_ms = (time.perf_counter() - t0) * 1000

        answer = result["answers"][qid]
        pred_choice = str(answer["choice"])
        p = normalize_probs(answer.get("probabilities", {}), keys)
        g = normalize_probs(row["gold"]["probabilities"], keys)
        gold_choice = keys[int(np.argmax(g))]

        soft_acc.append(float(np.sum(p * g)))
        brier.append(float(np.sum((p - g) ** 2)))
        tv.append(float(0.5 * np.sum(np.abs(p - g))))
        kl.append(
            float(np.sum(g * np.log(np.clip(g / np.clip(p, 1e-12, None), 1e-12, 1e12))))
        )
        latencies.append(latency_ms)

        predictions.append(
            {
                "test_number": n,
                "jsonl_row": int(row_idx),
                "question_id": qid,
                "state": row["state"],
                "question": qdef,
                "prediction": {
                    "choice": pred_choice,
                    "probabilities": {k: float(p[j]) for j, k in enumerate(keys)},
                    "confidence": float(p.max()),
                },
                "gold": {
                    "choice": gold_choice,
                    "probabilities": {k: float(g[j]) for j, k in enumerate(keys)},
                },
                "correct": pred_choice == gold_choice,
                "latency_ms": float(latency_ms),
            }
        )

    n_correct = sum(x["correct"] for x in predictions)
    metrics = {
        "model": model_name,
        "dataset": JSONL_PATH,
        "n_examples": len(predictions),
        "correct": n_correct,
        "wrong": len(predictions) - n_correct,
        "accuracy": n_correct / len(predictions),
        "soft_accuracy": float(np.mean(soft_acc)),
        "brier_score": float(np.mean(brier)),
        "kl_divergence": float(np.mean(kl)),
        "tv_distance": float(np.mean(tv)),
        "latency_mean_ms": float(np.mean(latencies)),
        "latency_p50_ms": float(np.percentile(latencies, 50)),
        "latency_p95_ms": float(np.percentile(latencies, 95)),
        "total_inference_seconds": time.time() - t_start,
    }
    return metrics, predictions


def print_metrics(m):
    """Pretty-print the metrics dict returned by `evaluate_agent`."""
    print(f"Model          : {m['model']}")
    print(
        f"Examples       : {m['n_examples']} (correct {m['correct']}, wrong {m['wrong']})"
    )
    print(f"Accuracy       : {m['accuracy']:.4f} ({m['accuracy'] * 100:.2f}%)")
    print(f"Soft accuracy  : {m['soft_accuracy']:.6f}")
    print(f"Brier score    : {m['brier_score']:.6f}")
    print(f"KL divergence  : {m['kl_divergence']:.6f}")
    print(f"TV distance    : {m['tv_distance']:.6f}")
    print(
        f"Latency (ms)   : mean {m['latency_mean_ms']:.2f} | "
        f"p50 {m['latency_p50_ms']:.2f} | p95 {m['latency_p95_ms']:.2f}"
    )
    print(f"Total inference: {m['total_inference_seconds']:.2f} s")

In [13]:
test_indices = test_idx[:BASELINE_N]
if len(test_indices) < BASELINE_N:
    raise ValueError(f"Only {len(test_indices)} test rows available, need {BASELINE_N}")
test_rows = [rows[i] for i in test_indices]

print(f"Loading base Laya and scoring {len(test_rows)} held-out test rows...")
agent_base = laya.Agent(model_dir, device="cuda")

baseline_metrics, baseline_predictions = evaluate_agent(
    agent_base, test_rows, test_indices, "Laya base - before fine-tuning"
)

BASELINE_RESULTS_PATH.write_text(
    json.dumps(
        {"metrics": baseline_metrics, "predictions": baseline_predictions},
        indent=2,
        ensure_ascii=False,
    )
)

print("\n" + "=" * 80)
print_metrics(baseline_metrics)
print(f"\n✓ Results saved to {BASELINE_RESULTS_PATH}")

# Release the GPU memory held by the base agent so the DDP workers in Step 9
# get the full T4s.
del agent_base
gc.collect()
torch.cuda.empty_cache()

Loading base Laya and scoring 1000 held-out test rows...


/tmp/ipykernel_22/315701730.py:7: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent_base = laya.Agent(model_dir, device="cuda")



Model          : Laya base - before fine-tuning
Examples       : 1000 (correct 261, wrong 739)
Accuracy       : 0.2610 (26.10%)
Soft accuracy  : 0.290975
Brier score    : 0.411140
KL divergence  : 0.924530
TV distance    : 0.520754
Latency (ms)   : mean 35.64 | p50 34.32 | p95 36.75
Total inference: 35.73 s

✓ Results saved to /kaggle/working/laya_baseline_1000_before_finetuning.json


## Step 7 — Save the train and validation items

The DDP training script runs in separate processes (one per GPU), so it cannot see variables from this notebook. The tokenised **train** items are therefore written to disk, together with the validation items (kept for later evaluation; the training script does not use them).

Each file is re-loaded once to confirm it was written correctly. The test split is not saved: it is addressed through `test_idx` and `rows`.

In [14]:
train_items = [examples[i] for i in train_idx]
val_items = [examples[i] for i in val_idx]

torch.save(train_items, TRAIN_ITEMS_PATH)
torch.save(val_items, VAL_ITEMS_PATH)

for path, expected in [
    (TRAIN_ITEMS_PATH, len(train_items)),
    (VAL_ITEMS_PATH, len(val_items)),
]:
    assert path.exists()
    assert len(torch.load(path, weights_only=False)) == expected

print(f"✓ Train items      : {len(train_items):,} -> {TRAIN_ITEMS_PATH}")
print(f"✓ Validation items : {len(val_items):,} -> {VAL_ITEMS_PATH}")
print(
    f"  Test rows        : {len(test_idx):,} (kept separate, never saved for training)"
)

✓ Train items      : 120,305 -> /kaggle/working/train_items.pt
✓ Validation items : 15,038 -> /kaggle/working/validation_items.pt
  Test rows        : 15,039 (kept separate, never saved for training)


## Step 8 — Write the DDP fine-tuning script

The next cell writes `train_ddp.py`, which `torchrun` starts once per GPU. It follows Laya's reference fine-tuning recipe, with the epoch count reduced from 4 to 1 to fit a Kaggle session.

**What the script does**

1. **Setup:** initialises the NCCL process group, builds the base model from `model_dir`, loads the pretrained weights, enables gradient checkpointing and wraps the model in DDP.
2. **Data:** loads the train items, holds out up to **400 items for calibration**, and shards the rest evenly across the GPUs.
3. **Optimiser:** AdamW with two learning rates (encoder `2.5e-5`, head `1e-4`), cosine learning-rate decay, fp16 autocast with a gradient scaler and gradient clipping at 1.0.
4. **Loss for every micro-batch**, which combines two terms:
   - *Reinforcement term:* several noisy variants of the predicted logits are sampled, each is scored against the gold distribution with a proper scoring reward (spherical + ranked probability score), and the group-normalised advantage weights the log-probability of the noise.
   - *Supervised term:* soft cross-entropy between the predicted and gold distributions.
5. **Checkpoint:** a rolling fp16 checkpoint is saved after each epoch.
6. **Calibration:** after training, one softmax **temperature** per question type (`choice`, `score`, `noul`) is fitted on the held-out calibration items and stored in the saved config.
7. **Export:** writes weights, encoder config, tokenizer and `rl_agent_config.json` to the output directory.

**Hyper-parameters**

| Setting | Value |
|---|---|
| Epochs | 1 |
| Micro-batch / gradient accumulation | 8 / 4 (effective batch 64 across 2 GPUs) |
| Samples per item for the RL term (`GROUP_SIZE`) | 4 |
| Noise `sigma` | 0.4 → 0.1 over epochs (stays at 0.4 with a single epoch) |
| Reward weights | spherical 0.75, RPS 1.0 |

The three paths are passed as command-line arguments (model dir, output dir, train items file), so nothing is hard-coded in the script. `%%writefile` cannot read notebook variables, so if you change `TRAIN_SCRIPT_PATH` in the configuration, change the path on the first line of the cell as well.

> The base config tokenises with `max_len=512` / `head_max_len=192` (Step 4), while the script sets `1024` / `256` for model construction. This does no harm to the chess examples, which are only about 80 tokens long, because the script uses the pre-tokenised ids from Step 7 as they are.

In [15]:
%%writefile /kaggle/working/train_ddp.py
"""DDP fine-tuning of Laya on the chess typed-decision examples.

Usage (launched by the notebook with torchrun, one process per GPU):

    torchrun --standalone --nproc_per_node=2 train_ddp.py MODEL_DIR OUTPUT_DIR TRAIN_ITEMS_PT

MODEL_DIR       local snapshot of the base Laya checkpoint
OUTPUT_DIR      where the fine-tuned model is written
TRAIN_ITEMS_PT  torch file with the pre-tokenised training items (see notebook, Step 7)
"""

import json
import os
import random
import sys
import time

import torch
import torch.distributed as dist
from safetensors.torch import load_file, save_file
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoTokenizer

from laya.common import TEMP_MAX, TEMP_MIN, build_model, proper_reward


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def collate_train_batch(items, pad_id):
    """Pad a list of training items into one batch of tensors."""
    n = len(items)
    L = max(len(it["ids"]) for it in items)  # longest token sequence
    kmax = max(len(it["markers"]) for it in items)  # most options in the batch

    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)  # option marker positions
    mmask = torch.zeros((n, kmax), dtype=torch.bool)  # True where an option exists
    target = torch.zeros((n, kmax), dtype=torch.float32)  # gold distribution

    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"], dtype=torch.long)
        att[i, : len(it["ids"])] = 1

        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"], dtype=torch.long)
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)

    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def fit_one_temp(sel):
    """Fit one softmax temperature on (logits, target) pairs by minimising soft CE."""
    if len(sel) < 10:
        return 1.0

    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)  # padded logits
    T = torch.zeros((len(sel), kmax))  # padded targets
    for i, (z, t) in enumerate(sel):
        Z[i, : len(z)] = torch.tensor(z)
        T[i, : len(t)] = torch.tensor(t, dtype=torch.float32)

    log_t = torch.zeros(1, requires_grad=True)  # optimise log-temperature
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.clamp(log_t.exp(), TEMP_MIN, TEMP_MAX).item())


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    # --- Distributed setup -------------------------------------------------
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    train_items_path = sys.argv[3]

    # --- Config, tokenizer, model -----------------------------------------
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)

    # Settings taken from Laya's reference fine-tuning notebook
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))

    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(
        load_file(os.path.join(model_dir, "model.safetensors")), strict=True
    )

    model.encoder.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.head_checkpointing = True

    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    # --- Data --------------------------------------------------------------
    all_items = torch.load(train_items_path, weights_only=False)
    assert len(all_items) > 0, f"No training items found in {train_items_path}"

    if rank == 0:
        print()
        print("=" * 80)
        print("LAYA DDP FINE-TUNING")
        print("=" * 80)
        print(f"Training examples : {len(all_items):,}")
        print(f"World size        : {world_size}")
        print("Epochs            : 2")
        print("Validation set    : NOT used for training")
        print("Test set          : NOT used for training")

    # Hold out a small calibration set (used after training to fit temperatures)
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)

    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]

    # Same number of items on every GPU, then shard round-robin
    train_items = train_items[: len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]

    if rank == 0:
        print(f"Calibration holdout : {len(calib_items):,}")
        print(f"Training after calibration : {len(train_items):,}")
        print(f"Training per GPU : {len(my_items):,}")

    # --- Hyper-parameters (Laya defaults; only EPOCHS changed from 4 to 2) --
    EPOCHS = 2
    MICRO_BATCH = 8  # items per forward/backward pass, per GPU
    GRAD_ACCUM = 4  # micro-batches per optimizer step
    GROUP_SIZE = 4  # noisy samples per item for the RL term
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4  # exploration noise, annealed linearly over epochs
    SIGMA_END = 0.1

    # --- Optimiser, scheduler, AMP scaler ---------------------------------
    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW(
        [
            {"params": enc_params, "lr": LR_ENCODER},
            {"params": head_params, "lr": LR_HEAD},
        ],
        weight_decay=0.01,
    )

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, total_updates), eta_min=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    # --- Training loop -----------------------------------------------------
    if rank == 0:
        print()
        print("Starting 2-epoch 2xT4 DDP training...")
        print()

    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)

        epoch_loss = 0.0
        n_batches = 0
        accum_step = 0
        optimizer.zero_grad(set_to_none=True)

        # With EPOCHS = 1 progress is 0, so sigma stays at SIGMA_START
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx : b_idx + MICRO_BATCH]
            if not chunk:
                continue

            batch = collate_train_batch(chunk, tok.pad_token_id)

            # Forward pass (fp16 autocast)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()  # options per item
            target = batch["target"].to(device)

            # Sample GROUP_SIZE noisy variants of the logits (noise is zero-mean
            # across options) and turn each into a probability distribution
            eps = (
                torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            )
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

            # Reward = proper scoring rule between sampled distribution and gold;
            # advantage = reward relative to the group mean, normalised
            with torch.no_grad():
                r = proper_reward(
                    q,
                    target.unsqueeze(0),
                    batch["qtype"].to(device),
                    mask,
                    w_sph=0.75,
                    w_rps=1.0,
                )
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            # Policy-gradient term: Gaussian log-prob of the injected noise
            logp = -((z - logits.unsqueeze(0)) ** 2 * mask).sum(-1) / (2 * sigma**2)
            loss_rl = -(adv * logp).mean()

            # Supervised term: soft cross-entropy against the gold distribution
            loss_ce = (
                -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1))
                .sum(-1)
                .mean()
            )

            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM
            # Keeps the action head in the autograd graph (required by DDP)
            loss = loss + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1

            # Optimiser step every GRAD_ACCUM micro-batches (and on the last one)
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1

            if rank == 0 and n_batches % 50 == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(
                    f"Epoch {epoch + 1}/{EPOCHS}"
                    f" | Step {n_batches}"
                    f" | Loss {loss.item() * GRAD_ACCUM:.4f}"
                    f" | Reward {r.mean().item():.3f}"
                    f" | LR {cur_lr:.2e}"
                )

        if rank == 0:
            print()
            print(
                f"=== Epoch {epoch + 1}/{EPOCHS}"
                f" Completed in {time.time() - t0:.1f}s"
                f" | Avg Loss: {epoch_loss / max(1, n_batches):.4f} ==="
            )

        dist.barrier()

        # Rolling checkpoint (rank 0 only, fp16)
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)

            ckpt_sd = {
                k: v.half().contiguous().cpu() for k, v in model.state_dict().items()
            }
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(ckpt_dir)

            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump(
                    {
                        "epoch": epoch + 1,
                        "total_epochs": EPOCHS,
                        "avg_loss": epoch_loss / max(1, n_batches),
                    },
                    f,
                    indent=2,
                )

            print("✓ Saved rolling checkpoint:", ckpt_dir)

    # --- Post-training calibration + final save (rank 0 only) --------------
    dist.barrier()

    if rank == 0:
        print()
        print("Fitting post-training calibration temperatures...")

        # Free optimiser memory before running inference
        del optimizer
        del scaler
        del scheduler
        torch.cuda.empty_cache()

        model.eval()
        calib_preds = []  # (question type, logits, target) per calibration item

        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx : c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)

                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device),
                    )

                l_np = l_sub.float().cpu().numpy()

                for r_idx, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r_idx, :k], it["target"]))

        # One temperature per question type: choice, score, noul
        fitted_temps = [1.2, 1.2, 1.2]  # fallback values

        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)

            print(
                "Fitted calibration temperatures (choice, score, noul):",
                [round(t, 3) for t in fitted_temps],
            )
        except Exception as e:
            print("Temperature fitting fallback:", e)

        # Final model: weights, encoder config, tokenizer, Laya config
        os.makedirs(output_dir, exist_ok=True)

        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(output_dir)

        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-chess"
        cfg["temperature"] = fitted_temps
        cfg.pop("temperature_by_options", None)

        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)

        print()
        print("✓ Final model saved to:", output_dir)

    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_ddp.py


## Step 9 — Launch fine-tuning on 2×T4

`torchrun` starts two worker processes (`--nproc_per_node=2`), one per GPU. Training progress is printed by rank 0 every 50 micro-batches. When it finishes, the fine-tuned model is in `OUTPUT_DIR` (with a `checkpoint_latest/` folder from the end of the epoch).

If you kept `MAX_ROWS` set for a smoke test, training runs on that smaller subset.

In [16]:
assert TRAIN_SCRIPT_PATH.exists(), "Run the Step 8 cell first to write train_ddp.py"

cmd = (
    f"torchrun --standalone --nproc_per_node=2 {TRAIN_SCRIPT_PATH} "
    f"{model_dir} {OUTPUT_DIR} {TRAIN_ITEMS_PATH}"
)

print("Executing DDP training:")
print(cmd, end="\n\n")

!{cmd}

Executing DDP training:
torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py /root/.cache/huggingface/hub/models--convaiinnovations--laya/snapshots/7b928d828b7b0e022f929d9bd2e44165aa270148 /kaggle/working/laya_finetuned_typed_decisions /kaggle/working/train_items.pt

W1004 12:40:20.273000 153 torch/distributed/run.py:851] 
W1004 12:40:20.273000 153 torch/distributed/run.py:851] *****************************************
W1004 12:40:20.273000 153 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1004 12:40:20.273000 153 torch/distributed/run.py:851] *****************************************

LAYA DDP FINE-TUNING
Training examples : 120,305
World size        : 2
Epochs            : 2
Validation set    : NOT used for training
Test set          : NOT used for training
Calibration holdou

### Outputs

| File | Created in | Description |
|---|---|---|
| `split_indices.json` | Step 5 | train / val / test row indices (reused on re-runs) |
| `laya_baseline_100_before_finetuning.json` | Step 6 | metrics and per-row predictions of the base model |
| `train_items.pt`, `validation_items.pt` | Step 7 | tokenised Laya items |
| `train_ddp.py` | Step 8 | DDP fine-tuning script |
| `laya_finetuned_typed_decisions/` | Step 9 | fine-tuned model, rolling checkpoint and calibrated config |

## Step 10 - Testing the final fine-tuned model

In [17]:
import shutil

FINETUNED_RESULTS_PATH = WORK_DIR / f"laya_finetuned_{BASELINE_N}_after_finetuning.json"

# The training script saves tokenizer files at the top of OUTPUT_DIR, but the
# base checkpoint keeps them in a "tokenizer/" subfolder. Mirror that layout.
tok_dir = OUTPUT_DIR / "tokenizer"
if not tok_dir.exists():
    tok_dir.mkdir()
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name not in {"model.safetensors", "rl_agent_config.json"}:
            shutil.copy(p, tok_dir)

# --- Score the fine-tuned model on the SAME test rows as the baseline ---
print(f"Loading fine-tuned Laya from {OUTPUT_DIR} ...")
agent_ft = laya.Agent(str(OUTPUT_DIR), device="cuda")

ft_metrics, ft_predictions = evaluate_agent(
    agent_ft, test_rows, test_indices, "Laya fine-tuned - chess typed decisions"
)

FINETUNED_RESULTS_PATH.write_text(
    json.dumps(
        {"metrics": ft_metrics, "predictions": ft_predictions},
        indent=2,
        ensure_ascii=False,
    )
)

print("\n" + "=" * 80)
print_metrics(ft_metrics)
print(f"\n✓ Results saved to {FINETUNED_RESULTS_PATH}")

# --- Before vs after (reload the baseline if the kernel was restarted) ---
if "baseline_metrics" not in globals():
    baseline_metrics = json.loads(BASELINE_RESULTS_PATH.read_text())["metrics"]

rows_cmp = [
    ("Accuracy", "accuracy", "higher"),
    ("Soft accuracy", "soft_accuracy", "higher"),
    ("Brier score", "brier_score", "lower"),
    ("KL divergence", "kl_divergence", "lower"),
    ("TV distance", "tv_distance", "lower"),
    ("Mean latency (ms)", "latency_mean_ms", "lower"),
]

print("\n" + "=" * 80)
print(f"{'Metric':<20}{'Base':>12}{'Fine-tuned':>14}{'Change':>12}   Better")
print("-" * 80)
for label, key, better in rows_cmp:
    b, f = baseline_metrics[key], ft_metrics[key]
    improved = (f > b) if better == "higher" else (f < b)
    print(f"{label:<20}{b:>12.4f}{f:>14.4f}{f - b:>+12.4f}   {'✓' if improved else '✗'} ({better})")

Loading fine-tuned Laya from /kaggle/working/laya_finetuned_typed_decisions ...

Model          : Laya fine-tuned - chess typed decisions
Examples       : 1000 (correct 497, wrong 503)
Accuracy       : 0.4970 (49.70%)
Soft accuracy  : 0.372064
Brier score    : 0.268018
KL divergence  : 0.544151
TV distance    : 0.396070
Latency (ms)   : mean 34.20 | p50 33.80 | p95 36.92
Total inference: 34.30 s

✓ Results saved to /kaggle/working/laya_finetuned_1000_after_finetuning.json

Metric                      Base    Fine-tuned      Change   Better
--------------------------------------------------------------------------------
Accuracy                  0.2610        0.4970     +0.2360   ✓ (higher)
Soft accuracy             0.2910        0.3721     +0.0811   ✓ (higher)
Brier score               0.4111        0.2680     -0.1431   ✓ (lower)
KL divergence             0.9245        0.5442     -0.3804   ✓ (lower)
TV distance               0.5208        0.3961     -0.1247   ✓ (lower)
Mean latency (ms